# 04 - Experiments And Improvements

This notebook defines the experiment plan. Training is disabled by default. It uses normal imports to inspect project code, and uses `subprocess` only for commands that would launch full scripts.

## Step 1 - Setup

In [43]:
from pathlib import Path
import inspect
import json
import sys

def find_notebooks_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd / "salient-object-detection-cnn" / "notebooks"]
    for parent in cwd.parents:
        candidates.extend([parent / "notebooks", parent / "salient-object-detection-cnn" / "notebooks"])
    for candidate in candidates:
        if (candidate / "notebook_utils.py").exists():
            return candidate
    raise FileNotFoundError("Could not find notebooks/notebook_utils.py")

NOTEBOOKS_DIR = find_notebooks_dir()
sys.path.insert(0, str(NOTEBOOKS_DIR))

from notebook_utils import PROJECT_DIR, PYTHON, run_script_live

sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project directory: {PROJECT_DIR}")

Using project directory: /Users/nafiesallahu/Documents/Nafja's Doc/Genpact/salient-object-detection-cnn


## Step 2 - Safety Switch

`RUN_TRAINING` is intentionally `False`. With this setting, training/evaluation/visualization cells only print the subprocess command to run later.

In [ ]:
RUN_TRAINING = True

def show_or_run(script_name, *args):
    command = [PYTHON, script_name, *map(str, args)]
    if RUN_TRAINING:
        return run_script_live(script_name, *args)
    print("DRY RUN - training is disabled")
    print("Command to run later:")
    print(" ".join(command))
    return None

## Step 3 - Experiment Plan

| Run | Purpose | Current support in code | Notes |
|---|---|---|---|
| Baseline | Original encoder-decoder model | Supported | Uses `--model_type baseline`, light augmentation, and the default loss weights. |
| Improved | Meets the assignment requirement after the baseline | Supported | Uses `unet_small` for deeper convolution blocks, skip connections, dropout, stronger augmentation, lower learning rate, and heavier IoU loss weighting. |
| Experiment 1 | Add Dropout | Supported | Use `--model_type unet_small --dropout 0.3`. |
| Experiment 2 | Add BatchNorm | Supported | Compare `baseline_no_bn` against `baseline` using the same settings. |
| Experiment 3 | Stronger augmentation | Supported | Use `--augmentation_strength strong`; this fixes the earlier issue where augmentation strength was hard-coded. |
| Experiment 4 | Different loss weighting | Supported | Use `--bce_weight` and `--iou_weight`. |


## Step 4 - Shared Experiment Settings

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 25
BASE_LR = 1e-3
IMPROVED_LR = 5e-4
NUM_WORKERS = 2
NUM_VISUALIZATION_SAMPLES = 10
DROPOUT = 0.3
BASE_IOU_WEIGHT = 0.5
IMPROVED_IOU_WEIGHT = 0.75

COMMON_TRAIN_ARGS = [
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--batch_size", BATCH_SIZE,
    "--epochs", EPOCHS,
    "--num_workers", NUM_WORKERS,
]

BASELINE_TRAIN_ARGS = [
    *COMMON_TRAIN_ARGS,
    "--lr", BASE_LR,
    "--model_type", "baseline",
    "--experiment_name", "baseline",
    "--augmentation_strength", "light",
    "--bce_weight", 1.0,
    "--iou_weight", BASE_IOU_WEIGHT,
]

IMPROVED_TRAIN_ARGS = [
    *COMMON_TRAIN_ARGS,
    "--lr", IMPROVED_LR,
    "--model_type", "unet_small",
    "--experiment_name", "improved",
    "--dropout", DROPOUT,
    "--augmentation_strength", "strong",
    "--bce_weight", 1.0,
    "--iou_weight", IMPROVED_IOU_WEIGHT,
]

COMMON_EVAL_ARGS = [
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--batch_size", BATCH_SIZE,
    "--num_workers", NUM_WORKERS,
]

print("Training will run:", RUN_TRAINING)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Baseline learning rate:", BASE_LR)
print("Improved learning rate:", IMPROVED_LR)


## Step 5 - Preflight: Inspect Current Implementation

In [ ]:
import importlib
import torch.nn as nn

import data_loader
import losses
import sod_model

# Jupyter can keep old module objects in memory after code edits.
# Reload here so this preflight cell always sees the current project code.
data_loader = importlib.reload(data_loader)
losses = importlib.reload(losses)
sod_model = importlib.reload(sod_model)

AUGMENTATION_STRENGTHS = data_loader.AUGMENTATION_STRENGTHS
BaselineSODCNN = sod_model.BaselineSODCNN
BaselineSODCNNNoBatchNorm = sod_model.BaselineSODCNNNoBatchNorm
SmallUNet = sod_model.SmallUNet
combined_loss = losses.combined_loss

for name, model in [
    ("baseline_no_bn", BaselineSODCNNNoBatchNorm()),
    ("baseline", BaselineSODCNN()),
    ("improved", SmallUNet(dropout=DROPOUT)),
]:
    conv_count = sum(1 for module in model.modules() if isinstance(module, nn.Conv2d))
    batchnorm_count = sum(1 for module in model.modules() if isinstance(module, nn.BatchNorm2d))
    dropout_count = sum(1 for module in model.modules() if isinstance(module, nn.Dropout2d))
    print(
        f"{name}: Conv2d layers={conv_count}, "
        f"BatchNorm2d layers={batchnorm_count}, "
        f"Dropout2d layers={dropout_count}"
    )

print("SmallUNet signature:", inspect.signature(SmallUNet))
print("combined_loss signature:", inspect.signature(combined_loss))
print("Augmentation strengths:", list(AUGMENTATION_STRENGTHS))


## Baseline - Original Model

In [ ]:
show_or_run("train.py", *BASELINE_TRAIN_ARGS)


### Baseline Evaluation And Visualization Commands

In [ ]:
show_or_run(
    "evaluate.py",
    *COMMON_EVAL_ARGS,
    "--model_type", "baseline",
    "--checkpoint", "checkpoints/best_model_baseline.pth",
    "--experiment_name", "baseline",
)

show_or_run(
    "visualize.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--model_type", "baseline",
    "--checkpoint", "checkpoints/best_model_baseline.pth",
    "--experiment_name", "baseline",
    "--num_samples", NUM_VISUALIZATION_SAMPLES,
)


## Improved Model - Multiple Changes

Purpose: satisfy the post-baseline requirement by changing the model/training setup in at least two ways.

This improved run changes the architecture to `unet_small`, uses deeper convolution blocks with skip connections, enables stronger dropout, applies stronger augmentation, lowers the learning rate, and increases the IoU loss weight.


In [ ]:
show_or_run("train.py", *IMPROVED_TRAIN_ARGS)


### Improved Evaluation And Visualization Commands


In [ ]:
show_or_run(
    "evaluate.py",
    *COMMON_EVAL_ARGS,
    "--model_type", "unet_small",
    "--checkpoint", "checkpoints/best_model_improved.pth",
    "--experiment_name", "improved",
)

show_or_run(
    "visualize.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--model_type", "unet_small",
    "--checkpoint", "checkpoints/best_model_improved.pth",
    "--experiment_name", "improved",
    "--num_samples", NUM_VISUALIZATION_SAMPLES,
)


## Experiment 2 - Add BatchNorm

Purpose: test whether BatchNorm stabilizes the baseline encoder-decoder model.

This is now a clean ablation: `baseline_no_bn` removes every BatchNorm layer, while `baseline` keeps the same architecture with BatchNorm. Keep all other training settings the same.


In [ ]:
from sod_model import BaselineSODCNN, BaselineSODCNNNoBatchNorm

print("BatchNorm ablation layer counts:")
for name, model in [("baseline_no_bn", BaselineSODCNNNoBatchNorm()), ("baseline", BaselineSODCNN())]:
    batchnorm_count = sum(1 for module in model.modules() if isinstance(module, nn.BatchNorm2d))
    print(f"{name}: BatchNorm2d layers={batchnorm_count}")

# Train/evaluate the no-BatchNorm baseline.
show_or_run(
    "train.py",
    *COMMON_TRAIN_ARGS,
    "--lr", BASE_LR,
    "--model_type", "baseline_no_bn",
    "--experiment_name", "baseline_no_bn",
    "--augmentation_strength", "light",
    "--bce_weight", 1.0,
    "--iou_weight", BASE_IOU_WEIGHT,
)

show_or_run(
    "evaluate.py",
    *COMMON_EVAL_ARGS,
    "--model_type", "baseline_no_bn",
    "--checkpoint", "checkpoints/best_model_baseline_no_bn.pth",
    "--experiment_name", "baseline_no_bn",
)

show_or_run(
    "visualize.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--model_type", "baseline_no_bn",
    "--checkpoint", "checkpoints/best_model_baseline_no_bn.pth",
    "--experiment_name", "baseline_no_bn",
    "--num_samples", NUM_VISUALIZATION_SAMPLES,
)

# Train/evaluate the same baseline with BatchNorm for the direct comparison.
show_or_run(
    "train.py",
    *BASELINE_TRAIN_ARGS,
)

show_or_run(
    "evaluate.py",
    *COMMON_EVAL_ARGS,
    "--model_type", "baseline",
    "--checkpoint", "checkpoints/best_model_baseline.pth",
    "--experiment_name", "baseline",
)

metric_files = {
    "baseline_no_bn": PROJECT_DIR / "outputs/metrics/test_metrics_baseline_no_bn.json",
    "baseline_with_bn": PROJECT_DIR / "outputs/metrics/test_metrics_baseline.json",
}

header = ["experiment", "iou", "precision", "recall", "f1_score", "mae", "mse", "status"]
print("\nBatchNorm comparison")
print("\t".join(header))
for name, path in metric_files.items():
    if path.exists():
        metrics = json.loads(path.read_text())
        values = [name] + [f"{metrics[key]:.4f}" for key in header[1:-1]] + ["completed"]
    else:
        values = [name, "-", "-", "-", "-", "-", "-", f"missing: {path.name}"]
    print("\t".join(values))


## Experiment 3 - Stronger Augmentation

Purpose: test whether stronger augmentation improves robustness on unseen DUTS-TE images.

Current status: supported. The earlier mistake was that augmentation strength was hard-coded; it is now configurable with `--augmentation_strength none|light|strong`.


In [ ]:
from data_loader import PreprocessedDUTSDataset

print("Light augmentation config:", AUGMENTATION_STRENGTHS["light"])
print("Strong augmentation config:", AUGMENTATION_STRENGTHS["strong"])
print(inspect.getsource(PreprocessedDUTSDataset._apply_train_augmentations))

show_or_run(
    "train.py",
    *COMMON_TRAIN_ARGS,
    "--lr", BASE_LR,
    "--model_type", "baseline",
    "--experiment_name", "strong_aug",
    "--augmentation_strength", "strong",
    "--bce_weight", 1.0,
    "--iou_weight", BASE_IOU_WEIGHT,
)


## Experiment 4 - Different Loss Weighting

Purpose: test whether emphasizing IoU improves mask overlap compared with the default loss.

Current status: supported with `--bce_weight` and `--iou_weight`.


In [ ]:
print(inspect.getsource(combined_loss))

loss_weight_plan = [
    {"name": "baseline", "bce_weight": 1.0, "iou_weight": 0.5},
    {"name": "iou_heavy", "bce_weight": 1.0, "iou_weight": 1.0},
]

for plan in loss_weight_plan:
    print(plan)

show_or_run(
    "train.py",
    *COMMON_TRAIN_ARGS,
    "--lr", BASE_LR,
    "--model_type", "baseline",
    "--experiment_name", "iou_heavy",
    "--augmentation_strength", "light",
    "--bce_weight", 1.0,
    "--iou_weight", 1.0,
)


## Step 6 - Results Table And Visual Comparison


In [ ]:
metric_files = {
    "baseline": PROJECT_DIR / "outputs/metrics/test_metrics_baseline.json",
    "improved": PROJECT_DIR / "outputs/metrics/test_metrics_improved.json",
    "strong_aug": PROJECT_DIR / "outputs/metrics/test_metrics_strong_aug.json",
    "iou_heavy": PROJECT_DIR / "outputs/metrics/test_metrics_iou_heavy.json",
}

header = ["experiment", "iou", "precision", "recall", "f1_score", "mae", "mse", "status"]
print("	".join(header))
for name, path in metric_files.items():
    if path.exists():
        metrics = json.loads(path.read_text())
        values = [name] + [f"{metrics[key]:.4f}" for key in header[1:-1]] + ["completed"]
    else:
        values = [name, "-", "-", "-", "-", "-", "-", f"missing: {path.name}"]
    print("	".join(values))

show_or_run("compare_experiments.py", "--baseline_name", "baseline", "--improved_name", "improved")


## Step 7 - Fair Comparison Checklist

- Use the same DUTS train/validation/test split for every run.
- Use the same image size, batch size, epoch count, optimizer, and random seed unless that is the variable being tested.
- Save a separate checkpoint and metric JSON for each experiment with `--experiment_name`.
- Include at least 3 to 5 visual examples per experiment with `visualize.py`.
- Report both quantitative results and qualitative failure cases.
